In [ ]:
import jax.numpy as jnp
import roughpy_jax as rpj
import numpy as np
import jax
from roughpy_jax.streams import LieIncrementStream
from roughpy_jax.streams.lie_increment_stream import _zero_lie
from roughpy_jax.intervals import IntervalType, Partition
from roughpy_jax.streams.piecewise_abelian_stream import to_piecewise_abelian_stream
from roughpy_jax.algebra import to_signature, antipode, to_log_signature, lie_to_tensor, as_free_tensor, _remove_unit_term
from roughpy_jax.dense_algebra import get_batch_shape, _algebra_scalar_multiply, broadcast_to_batch_shape, identity_like
from utils import generate_sinusoidal_timeseries, uniform_intervals, to_list_format, sigs_over_intervals, make_incremental
from solver import RoughKernel

In [ ]:
rk = RoughKernel(5, 5)
intervals = uniform_intervals(5)

In [ ]:
B = 10
N = 50
W = 2
key = jax.random.PRNGKey(0)

times, data = generate_sinusoidal_timeseries(key, B, N, W)
_, data_inc = make_incremental(times, data)
times, data = to_list_format(times[:, 1:], data_inc)

In [ ]:
K = rk.solve_PDE(data, times, intervals)

TracerArrayConversionError: The numpy.ndarray conversion method __array__() was called on traced array with shape float32[55,63]
The error occurred while tracing the function compute_phi at c:\coding\roughsigkernel\solver.py:85 for jit. This concrete value was not available in Python because it depends on the values of the arguments xi[0] and psi01[0].
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerArrayConversionError

In [16]:
K[-1, -1]

Array([ 1.5906441e+01,  7.3840761e+00,  5.9046249e+01, -5.5271745e+00,
       -1.9421281e+02, -1.2033177e+01,  2.5501518e+00, -7.9099650e+00,
        1.5683506e+01,  1.9352500e+00,  8.6862152e+02, -1.2993076e+01,
       -1.3416306e+01,  2.2222931e+02, -1.0835867e+01, -4.2447166e+01,
       -2.6301804e+01,  5.5585527e+00,  9.3786163e+00,  1.2614685e+02,
       -9.7184696e+00,  1.8028607e+02,  3.6238004e+02,  9.4200554e+00,
       -7.4640350e+00, -3.8165428e+01, -1.3784176e+01,  7.2854942e+01,
        7.4147391e+00, -3.7863277e+01,  1.8660744e+01, -5.6417459e-01,
       -5.5384226e+00,  7.5862398e+00,  4.6517500e+05,  3.9123413e+01,
        6.6092205e+00, -2.5740906e+01, -4.5927940e+01, -3.6327533e+02,
        4.0426099e+02,  8.3203688e+00, -3.8363792e+01,  2.3975208e+01,
       -1.1538948e+01,  2.1733170e+02, -6.9810185e+00,  1.8504105e+00,
       -1.6404629e+01,  2.2415366e+02,  8.2168282e+01, -1.5046612e+01,
        1.6060449e+03,  9.6616421e+00,  1.3374863e+03], dtype=float32)

In [6]:
K[-1, -1]

Array([ 1.3373649e+00, -1.4146852e+00, -1.5133532e+01,  2.8814816e+00,
        2.4204540e+00,  4.0481329e+00,  2.1406252e+00, -5.5146275e+00,
       -7.3349309e-01,  5.8950338e+00, -3.9521286e+01, -5.9496651e+00,
       -4.9474415e+01,  2.0726847e+01, -5.2058563e+00, -2.3476644e-01,
        8.0766907e+00, -8.2896090e+00, -1.4599148e+01, -7.6448982e+01,
       -6.0935736e+00, -8.7586975e-01, -2.3583813e+02,  8.4865397e-01,
       -7.7107973e+00,  5.2807169e+00,  9.0946007e+00,  9.1323395e+00,
        1.1817627e+01, -3.9102502e+00,  6.5852302e-01, -3.2060339e+00,
       -5.9031496e+00,  1.2969982e+01, -2.0480830e+03,  1.9651630e+01,
       -3.3331106e+00,  6.3886127e+01,  8.8761749e+01,  6.1529274e+00,
        2.2329980e+02, -3.5218432e+00, -2.9259977e+00,  1.3522969e+01,
        9.7414589e+00,  1.4724864e+00,  8.6757952e-01, -2.5299277e+00,
        6.1871438e+00,  1.0452487e+00, -1.2044094e+00,  5.4911656e+00,
        3.1665697e+00, -1.7007637e-01, -1.9022814e+01], dtype=float32)

In [7]:
K2 = solve_PDE(data, times, intervals, 3, 4)

In [3]:
R=4 
L=5
n=3
intervals = uniform_intervals(L)
K = solve_PDE(data=data, times=times, intervals=intervals, n=n, R=R)

In [6]:
K.shape

(6, 6, 55)

In [17]:
# pairwise comparison
Lie_Basis = rpj.LieBasis(depth = 5, width = W)
Tensor_Basis = rpj.to_tensor_basis(Lie_Basis)

X_Lie = LieIncrementStream.from_increments(timestamps=jnp.array(times[0]),
                                           data=data_inc[0],
                                           resolution=5,
                                           input_data_basis=None,
                                           lie_basis=Lie_Basis)

Y_Lie = LieIncrementStream.from_increments(timestamps=jnp.array(times[0]),
                                           data=data_inc[0],
                                           resolution=5,
                                           input_data_basis=None,
                                           lie_basis=Lie_Basis)

In [18]:
K[-1, -1, 0], rpj.tensor_pairing(X_Lie.signature(), Y_Lie.signature())

(Array(15.906441, dtype=float32), Array([18.488033], dtype=float32))

In [ ]:
xsig = X_Lie.signature()
ysig = Y_Lie.signature()

rpj.tensor_pairing(xsig, ysig)
identity_like(xsig).__array__()

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

In [ ]:
K

np.float32(-2.0578737)

In [ ]:
# checking one-to-zero and add-tensor-scalar

def add_tensor_scalar(a, s):
    cls = type(a)
    scalar = jnp.asarray(s)
    ext_scalar = broadcast_to_batch_shape(scalar, a.batch_shape)
    result_data = jnp.add(a.data, ext_scalar)
    return cls(result_data, a.basis)

# batched version
batch_sig
add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(batch_sig, batch_sig)), -batched_inner(xsig_one, ysig_one)).__array__()

# single version
xsig_one = X_Lie.signature()
ysig_one = Y_Lie.signature()
add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(xsig_one, ysig_one)), -batched_inner(xsig_one, ysig_one)).__array__()

array([[0.        , 1.2387185 , 0.31937182, 1.480955  , 0.39571154,
        1.5834085 , 1.1892676 , 1.3261325 , 0.77462494, 0.93321896,
        0.91428727, 1.3176042 , 1.5388111 , 0.79017377, 1.064494  ]],
      dtype=float32)